In [27]:
accounts = {}
current_account = None
next_account = 1001

In [28]:
def get_name():
    while True:
        name = input("Enter Name: ").strip().title() # strip() remove space / Title() first latter capital

       
        if name and "|" not in name:
            return name

        print("Invalid Name!")

In [29]:
def get_pin():
    while True:
        pin = input("Enter 4 Digit PIN: ")

        if pin.isdigit() and len(pin) == 4:  # isdigit() to check 0 to 9 numbers
            return pin

        print("PIN must be 4 digits!")


In [30]:
def get_mobile():
    while True:
        mobile = input("Enter Mobile Number: ").strip()

        if mobile.isdigit() and len(mobile) == 10:
            return mobile

        print("Invalid Mobile Number!")



In [31]:

def get_amount():
    while True:
        try:
            amount = float(input("Enter Amount: "))

            if amount > 0:
                return round(amount, 2)

            print("Amount must be positive!")

        except ValueError:
            print("Enter valid amount!")

In [32]:
def require_login():
    if current_account is None:
        print("Please Login First!")
        return False

    return True


In [33]:
def add_history(account_no, message):
    accounts[account_no]["history"].append(message)

# 2. Create Account

In [34]:

def create_account():

    global next_account

    print("\n===== CREATE ACCOUNT =====")

    name = get_name()
    pin = get_pin()
    mobile = get_mobile()

    while True:
        try:
            balance = float(input("Opening Balance: "))

            if balance >= 0:
                break

            print("Balance cannot be negative!")

        except ValueError:
            print("Enter valid balance!")

    account_no = str(next_account)
    next_account += 1

    accounts[account_no] = {
        "name": name,
        "pin": pin,
        "mobile": mobile,
        "balance": round(balance, 2),
        "history": []
    }

    add_history(
        account_no,
        f"Account Created: {balance:.2f}"
    )

    print("\nAccount Created Successfully!")
    print("Account Number:", account_no)

    save_data(False)

# Login

In [35]:

def login():

    global current_account

    print("\n===== LOGIN =====")

    current_account = None

    account_no = input("Account Number: ").strip()
    pin = input("Enter PIN: ")

    if account_no in accounts:

        if accounts[account_no]["pin"] == pin:

            current_account = account_no

            print("Login Successful!")
            print("Welcome", accounts[account_no]["name"])

        else:
            print("Wrong PIN!")

    else:
        print("Account Not Found!")

    

# Deposite Money

In [36]:
def deposit_money():

    if not require_login():
        return

    amount = get_amount()

    data = accounts[current_account]

    data["balance"] = round(data["balance"] + amount, 2)

    add_history(
        current_account,
        f"Deposit: {amount:.2f}"
    )

    print("Deposit Successful!")
    print("Balance:", data["balance"])

    save_data(False)


# 4. WITHDRAW MONEY

In [37]:
def withdraw_money():

    if not require_login():
        return

    amount = get_amount()

    data = accounts[current_account]

    if amount <= data["balance"]:

        data["balance"] = round(data["balance"] - amount, 2)

        add_history(
            current_account,
            f"Withdraw: {amount:.2f}"
        )

        print("Withdraw Successful!")
        print("Balance:", data["balance"])

        save_data(False)

    else:
        print("Insufficient Balance!")


# 5. CHECK BALANCE

In [38]:
def check_balance():

    if not require_login():
        return

    balance = accounts[current_account]["balance"]

    print("Available Balance:", balance)

#  6. TRANSFER MONEY


In [39]:

def transfer_money():

    if not require_login():
        return

    receiver = input("Receiver Account Number: ").strip()

    if receiver not in accounts:
        print("Receiver Not Found!")
        return

    if receiver == current_account:
        print("Cannot transfer to your own account!")
        return

    amount = get_amount()

    sender_data = accounts[current_account]
    receiver_data = accounts[receiver]

    if amount <= sender_data["balance"]:

        sender_data["balance"] = round(
            sender_data["balance"] - amount, 2
        )

        receiver_data["balance"] = round(
            receiver_data["balance"] + amount, 2
        )

        add_history(
            current_account,
            f"Sent {amount:.2f} to {receiver}"
        )

        add_history(
            receiver,
            f"Received {amount:.2f} from {current_account}"
        )

        print("Transfer Successful!")

        save_data(False)

    else:
        print("Insufficient Balance!")

# 8. TRANSACTION HISTORY

In [40]:

def transaction_history():

    if not require_login():
        return

    history = accounts[current_account]["history"]

    print("\n===== TRANSACTION HISTORY =====")

    if not history:
        print("No Transactions!")

    else:
        for number, transaction in enumerate(history, start=1):
            print(number, transaction)


# 9. ACCOUNT DETAILS

In [41]:
def account_details():

    if not require_login():
        return

    data = accounts[current_account]

    print("\n===== ACCOUNT DETAILS =====")

    for key, value in data.items():

        if key != "pin" and key != "history":
            print(key.title(), ":", value)


# 10. CHANGE PIN

In [42]:
def change_pin():

    if not require_login():
        return

    data = accounts[current_account]

    old_pin = input("Enter Old PIN: ")

    if old_pin == data["pin"]:

        new_pin = get_pin()

        if new_pin != old_pin:

            data["pin"] = new_pin

            print("PIN Changed Successfully!")

            save_data(False)

        else:
            print("New PIN must be different!")

    else:
        print("Wrong PIN!")

# 11. SEARCH ACCOUNT

In [43]:
def search_account():

    search = input("Enter Account Number or Name: ").strip()

    found = False

    for account_no, data in accounts.items():

        if (
            account_no == search
            or data["name"].lower() == search.lower()
        ):

            print("Account Found!")
            print("Account:", account_no)
            print("Name:", data["name"])
            print("Mobile:", data["mobile"])

            found = True

    if not found:
        print("Account Not Found!")


# 12. UPDATE ACCOUNT

In [44]:
def update_account():

    if not require_login():
        return

    data = accounts[current_account]

    print("1. Update Name")
    print("2. Update Mobile")

    choice = input("Enter Choice: ")

    if choice == "1":
        data["name"] = get_name()
        print("Name Updated!")

    elif choice == "2":
        data["mobile"] = get_mobile()
        print("Mobile Updated!")

    else:
        print("Invalid Choice!")
        return

    save_data(False)

# 13. DELETE ACCOUNT

In [45]:
def delete_account():

    global current_account

    if not require_login():
        return

    pin = input("Enter PIN: ")

    if pin == accounts[current_account]["pin"]:

        confirm = input("Confirm Delete (yes/no): ").lower()

        if confirm == "yes":

            accounts.pop(current_account)

            current_account = None

            print("Account Deleted!")

            save_data(False)

        else:
            print("Deletion Cancelled!")

    else:
        print("Wrong PIN!")


# 14. BANK SUMMARY

In [46]:
def bank_summary():

    print("\n===== BANK SUMMARY =====")

    total_accounts = len(accounts)

    total_balance = sum(
        data["balance"] for data in accounts.values()
    )

    balances = [
        data["balance"] for data in accounts.values()
    ]

    print("Total Accounts:", total_accounts)
    print("Total Balance:", round(total_balance, 2))

    if balances:
        print("Highest Balance:", max(balances))
        print("Lowest Balance:", min(balances))
        print("Average Balance:", round(total_balance / total_accounts, 2))

    else:
        print("No Accounts Available!")

# 15. SAVE DATA

In [47]:
def save_data(show=True):

    try:
        with open("bank_data.txt", "w") as file:

            for account_no, data in accounts.items():

                file.write(
                    f"{account_no}|{data['name']}|"
                    f"{data['pin']}|{data['mobile']}|"
                    f"{data['balance']}\n"
                )

                for history in data["history"]:
                    file.write(
                        f"H|{account_no}|{history}\n"
                    )

        if show:
            print("Data Saved Successfully!")

    except OSError:
        print("Unable to Save Data!")

# 16. LOAD DATA

In [48]:
def load_data():

    global next_account

    try:
        with open("bank_data.txt", "r") as file:

            for line in file:

                parts = line.rstrip("\n").split("|")

                if parts[0] == "H" and len(parts) == 3:

                    account_no = parts[1]

                    if account_no in accounts:
                        accounts[account_no]["history"].append(parts[2])

                elif len(parts) == 5:

                    account_no = parts[0]

                    accounts[account_no] = {
                        "name": parts[1],
                        "pin": parts[2],
                        "mobile": parts[3],
                        "balance": float(parts[4]),
                        "history": []
                    }

                    next_account = max(
                        next_account,
                        int(account_no) + 1
                    )

    except FileNotFoundError:
        pass

    except (OSError, ValueError):
        print("Error loading saved data!")


# 17. SORT ACCOUNTS

In [49]:
def sort_accounts():

    print("\n===== SORT ACCOUNTS =====")

    sorted_data = sorted(
        accounts.items(),
        key=lambda item: item[1]["balance"],
        reverse=True
    )

    for account_no, data in sorted_data:

        print(
            account_no,
            data["name"],
            data["balance"]
        )


# 18. MAIN MENU

In [ ]:

def main():

    load_data()

    while True:

        print("\n================================")
        print("     SMART BANK MANAGEMENT")
        print("================================")

        menu = [
            "Create Account",
            "Login",
            "Deposit Money",
            "Withdraw Money",
            "Check Balance",
            "Transfer Money",
            "Transaction History",
            "Account Details",
            "Change PIN",
            "Search Account",
            "Update Account",
            "Delete Account",
            "Bank Summary",
            "Save Data",
            "Sort Accounts",
            "Exit"
        ]

        for number, item in enumerate(menu, start=1):
            print(f"{number}. {item}")

        choice = input("\nEnter Your Choice: ")

        if choice == "1":
            create_account()

        elif choice == "2":
            login()

        elif choice == "3":
            deposit_money()

        elif choice == "4":
            withdraw_money()

        elif choice == "5":
            check_balance()

        elif choice == "6":
            transfer_money()

        elif choice == "7":
            transaction_history()

        elif choice == "8":
            account_details()

        elif choice == "9":
            change_pin()

        elif choice == "10":
            search_account()

        elif choice == "11":
            update_account()

        elif choice == "12":
            delete_account()

        elif choice == "13":
            bank_summary()

        elif choice == "14":
            save_data()

        elif choice == "15":
            sort_accounts()

        elif choice == "16":
            save_data(False)
            print("Thank You For Using Smart Bank!")
            break

        else:
            print("Invalid Choice!")
main()


     SMART BANK MANAGEMENT
1. Create Account
2. Login
3. Deposit Money
4. Withdraw Money
5. Check Balance
6. Transfer Money
7. Transaction History
8. Account Details
9. Change PIN
10. Search Account
11. Update Account
12. Delete Account
13. Bank Summary
14. Save Data
15. Sort Accounts
16. Exit



Enter Your Choice:  1



===== CREATE ACCOUNT =====


Enter Name:  Jayesh Solanki
Enter 4 Digit PIN:  2077
Enter Mobile Number:  1212112121
Opening Balance:  5000



Account Created Successfully!
Account Number: 1001

     SMART BANK MANAGEMENT
1. Create Account
2. Login
3. Deposit Money
4. Withdraw Money
5. Check Balance
6. Transfer Money
7. Transaction History
8. Account Details
9. Change PIN
10. Search Account
11. Update Account
12. Delete Account
13. Bank Summary
14. Save Data
15. Sort Accounts
16. Exit



Enter Your Choice:  2


... (17 outputs left out)

Enter 4 Digit PIN:  2078
Enter Mobile Number:  1313112121
Opening Balance:  6000



Account Created Successfully!
Account Number: 1002

     SMART BANK MANAGEMENT
1. Create Account
2. Login
3. Deposit Money
4. Withdraw Money
5. Check Balance
6. Transfer Money
7. Transaction History
8. Account Details
9. Change PIN
10. Search Account
11. Update Account
12. Delete Account
13. Bank Summary
14. Save Data
15. Sort Accounts
16. Exit



Enter Your Choice:  6
Receiver Account Number:  1001


Cannot transfer to your own account!

     SMART BANK MANAGEMENT
1. Create Account
2. Login
3. Deposit Money
4. Withdraw Money
5. Check Balance
6. Transfer Money
7. Transaction History
8. Account Details
9. Change PIN
10. Search Account
11. Update Account
12. Delete Account
13. Bank Summary
14. Save Data
15. Sort Accounts
16. Exit
